# import dataset for past 10 years

In [1]:
import yfinance as yf
import pandas as pd
import math


In [2]:
import os 
import sys

project_root = os.path.abspath("..")
sys.path.insert(0,project_root)
from Data.data_loader import load_data


In [3]:
data= load_data()

data.head()
# data = yf.download(
#         "AMZN",
#         start="2016-11-10",
#         end="2026-07-10"
#     )



[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
Date,,,,,
2016-11-10,37.118999,38.941502,35.884998,38.940498,254940000
2016-11-11,36.950500,37.162998,36.445000,36.786499,132456000
2016-11-14,35.953499,37.299999,35.505001,37.275501,146426000
2016-11-15,37.161999,37.339001,36.299500,36.500000,135116000
2016-11-16,37.324501,37.493500,36.780499,36.993999,72976000


In [4]:
data.isnull().any()


Price   Ticker
Close   AMZN      False
High    AMZN      False
Low     AMZN      False
Open    AMZN      False
Volume  AMZN      False
dtype: bool

In [5]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2426 entries, 2016-11-10 to 2026-07-09
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, AMZN)   2426 non-null   float64
 1   (High, AMZN)    2426 non-null   float64
 2   (Low, AMZN)     2426 non-null   float64
 3   (Open, AMZN)    2426 non-null   float64
 4   (Volume, AMZN)  2426 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 113.7 KB


In [6]:
data.describe()


Price,Close,High,Low,Open,Volume
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
count,2426.000000,2426.000000,2426.000000,2426.000000,2.426000e+03
mean,134.407469,135.987796,132.754242,134.439717,7.098120e+07
std,57.780917,58.456229,57.109772,57.813755,3.849126e+07
min,35.953499,37.162998,35.505001,36.500000,1.142050e+07
25%,88.970623,89.780252,87.960627,89.043627,4.570132e+07
50%,131.370003,132.990997,129.470001,131.180000,6.067600e+07
75%,174.704998,177.175495,173.346130,175.253754,8.479100e+07
max,274.989990,278.559998,272.380005,276.079987,3.313000e+08


### technical indicators

In [7]:
data.columns =  data.columns.droplevel(1)
print(data.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')


#### simple moving average

In [8]:
data['sma-5'] = data['Close'].rolling(window=5).mean()
data['sma-20'] = data['Close'].rolling(window=20).mean()
data['sma-50'] = data['Close'].rolling(window=50).mean()
data['Close-sma-50'] = data['Close']-data['sma-50']
data['sma_5-sma_20'] = data['sma-5']-data['sma-20']
data['Price > SMA_50'] =(data['Close']>data['sma-50']).astype(int)
data['slope_sma-20'] = data['sma-20'].diff()

#### Exponential moving average

| SMA                          | EMA                                     |
| ---------------------------- | --------------------------------------- |
| Equal weight to all prices   | More weight to recent prices            |
| Slower to react              | Faster to react                         |
| Simpler                      | Slightly more complex                   |
| Good for long-term smoothing | Good for detecting recent trend changes |


# Exponential Moving Average (EMA) Use Cases

| EMA | Time Horizon | Use Case |
|------|--------------|----------|
| **EMA_5** | Very Short-Term | Captures recent price movements quickly. Useful for intraday trading, scalping, and identifying immediate momentum. |
| **EMA_10** | Short-Term | Tracks short-term trends while reducing some market noise. Commonly used by swing traders. |
| **EMA_20** | One-Month Trend | Identifies the primary short-to-medium-term trend. Often used as a dynamic support/resistance level and in Bollinger Band analysis. |
| **EMA_50** | Medium-Term | Measures the medium-term trend (approximately 2–3 months). Frequently used by institutional traders to identify sustained trends. |
| **EMA_100** | Long-Term | Helps identify long-term trend direction and filters out short-term market fluctuations. |
| **EMA_200** | Very Long-Term | One of the most widely used trend indicators. A price above the 200 EMA is generally considered bullish, while a price below it is considered bearish. Often used as a market regime filter. |

## Common EMA Combinations

| Combination | Purpose |
|-------------|---------|
| **EMA_5 & EMA_20** | Detect short-term trend changes and momentum shifts. |
| **EMA_10 & EMA_50** | Identify medium-term trend reversals. |
| **EMA_12 & EMA_26** | Used to calculate the **MACD** indicator. |
| **EMA_20 & EMA_50** | Confirm trend strength and identify potential pullbacks. |
| **EMA_50 & EMA_200** | Identify long-term bullish or bearish market conditions (Golden Cross / Death Cross). |

## For Machine Learning (AI Hedge Fund)

Instead of using only one EMA, include multiple EMA features:

- `EMA_5`
- `EMA_10`
- `EMA_20`
- `EMA_50`
- `EMA_100`
- `EMA_200`

These features help the model learn:
- Short-term momentum
- Medium-term trend
- Long-term trend
- Trend strength
- Trend reversals
- Multi-timeframe market behavior

In [9]:
data['EMA_20'] = data['Close'].ewm(span=20,adjust = False).mean()
data['EMA_5'] = data['Close'].ewm(span=5,adjust=False).mean()
data['EMA_50'] = data['Close'].ewm(span=50,adjust=False).mean()
data['EMA_200'] = data['Close'].ewm(span=200,adjust=False).mean()

In [10]:
data.tail()

Price,Close,High,Low,Open,Volume,sma-5,sma-20,sma-50,Close-sma-50,sma_5-sma_20,Price > SMA_50,slope_sma-20,EMA_20,EMA_5,EMA_50,EMA_200
Date,,,,,,,,,,,,,,,,
2026-07-02,242.669998,246.720001,241.080002,241.610001,48246400,239.107999,240.245999,255.4168,-12.746802,-1.138000,0,-0.367500,242.141982,239.680773,245.555128,234.158303
2026-07-06,244.160004,246.039993,240.880005,243.800003,40044000,241.401999,239.764500,255.1928,-11.032797,1.637499,0,-0.481499,242.334175,241.173850,245.500417,234.257822
2026-07-07,245.979996,248.929993,242.699997,246.979996,40579600,242.569998,239.762000,255.0108,-9.030804,2.807999,0,-0.002500,242.681396,242.775899,245.519224,234.374461
2026-07-08,243.619995,244.800003,240.520004,244.270004,29653000,243.625998,239.681999,254.6034,-10.983405,3.943999,0,-0.080000,242.770786,243.057264,245.444745,234.466456
2026-07-09,247.039993,247.500000,238.250000,239.820007,38091700,244.693997,239.824499,254.3218,-7.281807,4.869498,0,0.142500,243.177377,244.384840,245.507304,234.591566


#### RSI

momentum  indicator used in technical analysis . it measures the speed and magnitude of a security's recent price changes to detect overbought or oversold conditions in the price of that security. The RSI displayed as an oscilator(line graph) on a scale of 0 to 100.

1. RSI >70 overbought condition .
2. RSI <30 or below indicates oversold condition . 

formula to calculate RSI:
RSI step one =100−[ 100(1+(Average gain/ Average loss))]

In [11]:
def calculate_RSI(data,period):
    # price_change (today_closing - previous_day_closing) is denoted by delta.
    delta = data.diff()

    #  separate gain(positive delta values) and losses (absolute value of negative delta)
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # calculate avg gain and avg loss
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()

    # calculate relative strength
    rs = avg_gain/avg_loss

    # calculate RSI
    return  100-(100/(1+rs))

In [12]:
data[f'RSI_7']= calculate_RSI(data['Close'],7)
data[f'RSI_14']= calculate_RSI(data['Close'],14)
data[f'RSI_21']= calculate_RSI(data['Close'],21)

### MACD


The MACD indicator is a technical analysis tool used to assess momentum by measuring the relationship between two moving averages of a security's price to guide trading decisions.

Key Takeaways
1. Moving average convergence/divergence (MACD) is a technical indicator used to help investors identify entry points for buying or selling. The indicator is often available on the best online brokerage platforms.
2. The MACD line is calculated by subtracting the 26-period exponential moving average (EMA) from the 12-period EMA.
3. The signal line is a nine-period EMA of the MACD line.
4. MACD is best used with daily periods, where the traditional settings of 26/12/9 days are the default.

Traders may buy the security when the MACD line crosses above the signal line and sell—or short—the security when the MACD line crosses below the signal line. MACD indicators can be interpreted in several ways, but the more common methods are crossovers, divergences, and rapid rises/falls.

#### MACD Formula
MACD=12-Period EMA − 26-Period EMA

| Feature     | Meaning                                                                                           |
| ----------- | ------------------------------------------------------------------------------------------------- |
| `EMA_12`    | Short-term trend                                                                                  |
| `EMA_26`    | Medium-term trend                                                                                 |
| `MACD`      | Difference between short- and medium-term trends (momentum)                                       |
| `Signal`    | Smoothed MACD used to identify momentum changes                                                   |
| `Histogram` | Strength of the momentum; positive means MACD is above the signal line, negative means it's below |


In [13]:
period_12 = data['Close'].ewm(span=12,adjust=False).mean()
period_26 = data['Close'].ewm(span=26,adjust=False).mean()
data['macd'] = period_12-period_26
# The signal line is a 9-period EMA of the MACD.
data['macd_signal_line'] =data['macd'].ewm(span=9,adjust=False).mean()
data['histogram'] = data['macd']-data['macd_signal_line']



#  Bollinger Bands 

## Think of it like a road with two fences

Imagine you're driving on a road.

-  The **road** is the **20-period SMA (Moving Average)**.
-  The **left and right fences** are the **Upper Band** and **Lower Band**.
-  The **price** moves between these fences.

```text
                Upper Band
==================================================

                     Price
                       *
                     *
                  *

------------------ 20-Period SMA ------------------

              *
           *

==================================================
                Lower Band
```

Normally, the price moves between the two bands.

---

# Components

## 1. Middle Band

The middle band is simply:

```text
20-Period Simple Moving Average (SMA)
```

---

## 2. Upper Band

```text
Upper Band = SMA + (2 × Standard Deviation)
```

---

## 3. Lower Band

```text
Lower Band = SMA - (2 × Standard Deviation)
```

---

# Example 1: Calm Market (Low Volatility)

The price isn't moving much.

```text
Upper Band
--------------------------------------

Price
      *   *   *   *

Middle Band (SMA)
--------------------------------------

Lower Band
--------------------------------------
```

### Inference

- Low volatility
- Small price movement
- Quiet market

---

# Example 2: Volatile Market (High Volatility)

The price starts moving a lot.

```text
Upper Band
------------------------------------------------------------

Price
      /\      /\      /\
     /  \    /  \    /  \
    /    \  /    \  /    \

Middle Band (SMA)
------------------------------------------------------------

Lower Band
------------------------------------------------------------
```

### Inference

- High volatility
- Big price movement
- Active market

---

# Price Near Upper Band

```text
Upper Band
--------------------------------------   ⭐ Price

Middle Band
--------------------------------------

Lower Band
--------------------------------------
```

### Inference

- Strong upward momentum
- Buyers are currently stronger

---

# Price Near Lower Band

```text
Upper Band
--------------------------------------

Middle Band
--------------------------------------

Lower Band
-------------------------------------- ⭐ Price
```

### Inference

- Strong downward momentum
- Sellers are currently stronger

---

# Bollinger Band Squeeze

The bands become very close.

```text
Upper
------------

Middle
------------

Lower
------------
```

### Inference

- Very low volatility
- Market is quiet
- A larger move may happen soon (up or down)

---

# Wide Bands

```text
Upper
---------------------------------------------

Middle
----------------------------

Lower
---------------------------------------------
```

### Inference

- High volatility
- Large price swings
- Market is very active

---

# Formula

```text
Middle Band = 20-period SMA

Upper Band = SMA + (2 × Standard Deviation)

Lower Band = SMA − (2 × Standard Deviation)
```

---

# What information does Bollinger Band give?

| Situation | What it tells us |
|-----------|------------------|
| Bands Narrow | Low volatility |
| Bands Wide | High volatility |
| Price near Upper Band | Strong upward momentum |
| Price near Lower Band | Strong downward momentum |
| Price around SMA | Neutral trend |

---

# Features for Machine Learning (AI Hedge Fund)

Instead of using only the bands, create features like:

```python
Upper_Band
Lower_Band
Band_Width = Upper_Band - Lower_Band
Price_Position = (Close - Lower_Band) / (Upper_Band - Lower_Band)
```

These features help the model understand:

- Market volatility
- Trend strength
- Whether the price is near the upper or lower boundary
- Whether the market is calm or highly active

In [14]:
# Compute 20-period SMA
data["SMA_20"] = data["Close"].rolling(20).mean()

# Compute 20-period Standard Deviation
data["STD_20"] = data["Close"].rolling(20).std(ddof=0)

# Compute Bollinger Bands
data["Upper_Band"] = data["SMA_20"] + 2 * data["STD_20"]
data["Lower_Band"] = data["SMA_20"] - 2 * data["STD_20"]

This tells the model where the current price lies within the Bollinger Bands:

0 → At the lower band
0.5 → Around the middle band
1 → At the upper band

In [15]:
data["Price_Position"] = (
    (data["Close"] - data["Lower_Band"]) /
    (data["Upper_Band"] - data["Lower_Band"])
)

In [16]:
data.head()

Price,Close,High,Low,Open,Volume,sma-5,sma-20,sma-50,Close-sma-50,sma_5-sma_20,...,RSI_14,RSI_21,macd,macd_signal_line,histogram,SMA_20,STD_20,Upper_Band,Lower_Band,Price_Position
Date,,,,,,,,,,,,,,,,,,,,,
2016-11-10,37.118999,38.941502,35.884998,38.940498,254940000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2016-11-11,36.950500,37.162998,36.445000,36.786499,132456000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.013442,-0.002688,-0.010753,NaN,NaN,NaN,NaN,NaN
2016-11-14,35.953499,37.299999,35.505001,37.275501,146426000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.103352,-0.022821,-0.080531,NaN,NaN,NaN,NaN,NaN
2016-11-15,37.161999,37.339001,36.299500,36.500000,135116000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.076213,-0.033499,-0.042713,NaN,NaN,NaN,NaN,NaN
2016-11-16,37.324501,37.493500,36.780499,36.993999,72976000,36.9019,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.041118,-0.035023,-0.006095,NaN,NaN,NaN,NaN,NaN


# 📈 Average True Range (ATR) - Easy Explanation

## What is ATR?

**ATR (Average True Range)** measures **market volatility**.

It tells us:

> **"How much does the stock usually move in one period?"**

ATR **does NOT tell the direction** of the price.

It only tells **how much the price is moving**.

- High ATR → High volatility
- Low ATR → Low volatility

---

# Think of it like Weather 🌦️

Imagine two cities.

### City A

Temperature changes:

```text
30°C
31°C
30°C
32°C
31°C
```

Very small changes.

➡️ Low volatility
➡️ Low ATR

---

### City B

Temperature changes:

```text
30°C
42°C
27°C
39°C
24°C
```

Huge changes every day.

➡️ High volatility
➡️ High ATR

Stock prices behave the same way.

---

# Example

Suppose today's stock data is

| Previous Close | High | Low |
|---------------:|-----:|----:|
|100|110|104|

The stock moved a lot today.

ATR tries to measure that movement.

---

# Step 1: Calculate True Range (TR)

For every period, calculate these three values.

### 1. High − Low

```text
High - Low
```

Measures today's movement.

---

### 2. |High − Previous Close|

```text
|High - Previous Close|
```

Captures upward price gaps.

---

### 3. |Low − Previous Close|

```text
|Low - Previous Close|
```

Captures downward price gaps.

---

# Step 2: True Range

Take the **maximum** of the above three values.

```text
TR = max(
    High - Low,
    |High - Previous Close|,
    |Low - Previous Close|
)
```

---

# Example

Previous Close

```text
100
```

Today's High

```text
110
```

Today's Low

```text
104
```

Calculate

```text
High - Low = 6

|High - Previous Close| = 10

|Low - Previous Close| = 4
```

True Range

```text
TR = max(6,10,4)

TR = 10
```

The stock actually moved **10 points**, because of the gap from yesterday's close.

---

# Step 3: Average True Range (ATR)

ATR is simply the moving average of the True Range.

For a 14-period ATR

```text
ATR = Average of the last 14 True Range values
```

---

# Formula

## True Range

```text
TR = max(
High - Low,
|High - Previous Close|,
|Low - Previous Close|
)
```

---

## ATR

```text
ATR = 14-period Moving Average of TR
```

---

# Graphical Example

## Low ATR

```text
Price

110 ───────────────

108 ──────*────────

106 ───────*───────

104 ─────*─────────

102 ──────*────────

100 ───────────────
```

Small movement.

➡️ Low ATR

---

## High ATR

```text
Price

130 ───────*────────

120 ────────────────

110 ──*─────────────

100 ────────────────

90 ───────────*─────

80 ────────────────
```

Large movement.

➡️ High ATR

---

# Interpretation

| ATR | Meaning |
|------|---------|
| Low ATR | Calm market |
| High ATR | Volatile market |
| Rising ATR | Volatility increasing |
| Falling ATR | Volatility decreasing |

---

# ATR Does NOT Tell

❌ Buy signal

❌ Sell signal

❌ Trend direction

It only measures **volatility**.

---

```

---

# Features for Machine Learning (AI Hedge Fund)

Instead of only using ATR, create multiple features.

```python
ATR_7
ATR_14
ATR_21
ATR_50
```

Derived features:

```python
ATR_Percentage = ATR_14 / Close

ATR_Change = ATR_14.diff()

ATR_Increasing = ATR_14 > ATR_14.shift(1)
```

---

# Why is ATR useful for AI?

ATR helps the model understand:

- Current market volatility
- Risk level
- Whether volatility is increasing or decreasing
- Position sizing
- Stop-loss distance
- Whether the market is calm or highly active

Unlike SMA or EMA, ATR **does not predict price direction**.

It only tells **how much the price is likely to move**, making it an important feature for quantitative trading and AI-based hedge fund models.

In [17]:
h_l_diff = data['High']-data['Low']
data['prev_close'] = data['Close'].shift(1)
h_prevclose_diff = data['High']-data['prev_close']
l_prevclose_diff = data['Low']-data['prev_close']
print(h_l_diff)
true_range = pd.concat([
    h_l_diff,h_prevclose_diff,l_prevclose_diff
],axis=1).max(axis=1)

data['ATR'] = true_range.rolling(window=14).mean()

Date
2016-11-10    3.056503
2016-11-11    0.717999
2016-11-14    1.794998
2016-11-15    1.039501
2016-11-16    0.713001
                ...   
2026-07-02    5.639999
2026-07-06    5.159988
2026-07-07    6.229996
2026-07-08    4.279999
2026-07-09    9.250000
Length: 2426, dtype: float64


In [18]:
data.head()

Price,Close,High,Low,Open,Volume,sma-5,sma-20,sma-50,Close-sma-50,sma_5-sma_20,...,macd,macd_signal_line,histogram,SMA_20,STD_20,Upper_Band,Lower_Band,Price_Position,prev_close,ATR
Date,,,,,,,,,,,,,,,,,,,,,
2016-11-10,37.118999,38.941502,35.884998,38.940498,254940000,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-11-11,36.950500,37.162998,36.445000,36.786499,132456000,NaN,NaN,NaN,NaN,NaN,...,-0.013442,-0.002688,-0.010753,NaN,NaN,NaN,NaN,NaN,37.118999,NaN
2016-11-14,35.953499,37.299999,35.505001,37.275501,146426000,NaN,NaN,NaN,NaN,NaN,...,-0.103352,-0.022821,-0.080531,NaN,NaN,NaN,NaN,NaN,36.950500,NaN
2016-11-15,37.161999,37.339001,36.299500,36.500000,135116000,NaN,NaN,NaN,NaN,NaN,...,-0.076213,-0.033499,-0.042713,NaN,NaN,NaN,NaN,NaN,35.953499,NaN
2016-11-16,37.324501,37.493500,36.780499,36.993999,72976000,36.9019,NaN,NaN,NaN,NaN,...,-0.041118,-0.035023,-0.006095,NaN,NaN,NaN,NaN,NaN,37.161999,NaN


In [19]:
data.shape

(2426, 29)